# Figure 1: Landscape of the Asian Immune Diversity Atlas and demographic determinants of immune cell composition

This notebook reproduces panels of **Figure 1** of the AIDA AIRR manuscript:

- **Fig. 1B, C** — UMAP of T/NK cells, colored by annotated subsets (B) and TCR reconstruction status (C)
- **Fig. 1D, E** — UMAP of B cells, colored by annotated subsets (D) and BCR reconstruction status (E)
- **Fig. 1F** — Ridge regression coefficients linking demographic factors to T cell subset frequencies; stacked bar plot of TCR reconstruction rates
- **Fig. 1G** — T cell subset proportions stratified by sex and age group
- **Fig. 1H** — Ridge regression coefficients linking demographic factors to B cell subset frequencies; stacked bar plot of BCR reconstruction rates
- **Fig. 1I** — B cell subset proportions stratified by sex and age group

Inputs (h5ad objects with annotated cells and per-donor metadata) are expected under `data/`.
Outputs (PDF panels) are written under `figures/`.


## Imports and global plotting settings

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

import numpy as np
import pandas as pd
import scipy as sp
import scipy.stats as stats
from scipy.sparse import csr_matrix

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.patheffects as path_effects
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

import scanpy as sc
import scanpy.external as sce
import anndata as ad
import sceleto2 as scjp  # in-house Scanpy helper

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, MaxAbsScaler as mabs
from sklearn.utils import resample
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
from adjustText import adjust_text
from tqdm import tqdm

%matplotlib inline
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, color_map='OrRd')
sc.logging.print_version_and_date()

plt.rcParams['pdf.fonttype'] = 42
sns.set_style('ticks', {'axes.edgecolor': 'black', 'axes.edgewidth': 2})
sns.set_context("paper", font_scale=1.3, rc={'patch.linewidth': 1})


## Color palettes and category orders (shared across figures)

In [ ]:
Ethnicity_order = ['Chinese', 'Malay', 'Indian', 'Japanese', 'Korean', 'Thai']
Ethnicity_colors = ["tomato", 'gold', "dodgerblue", "limegreen", "aquamarine", "orchid", 'gray']

sex_colors = ["#87CEFA", "#FB7C7C"]
age_colors = ["#E5F5E0", "#74C476", "#238B45", "#00441B"]

# T/NK cell subset order
T_anno2_order = [
    'T_CD4_Naive_SOX4', 'T_CD4_Naive', 'T_CD4_cTfh', 'T_CD4_Th1', 'T_CD4_Th2',
    'T_CD4_Th17', 'T_CD4_activated', 'T_CD4_CTL', 'T_CD4_Treg', 'T_CD4_IFN',
    'T_CD8_Naive_SOX4', 'T_CD8_Naive', 'T_CD8_TEM_GZMK', 'T_CD8_TEM_GZMB', 'T_CD8_KIR',
    'T_unc_MAIT', 'T_unc_gdT', 'T_unc_dnT', 'NK_CD56br', 'NK_CD56dim', 'NK_CD56dim_KLRC2'
]

T_anno2_colors = [
    "lightsteelblue", "#4a6fe3", "mediumorchid", "#bb7784", "darkorange",
    "#023fa5", "lightblue", "darkgreen", "#d6bcc0", "tan",
    "goldenrod", "yellowgreen", "lightcoral", "#d33f6a", "#11c638",
    "darkorchid", "#ef9708", "#0fcfc0", "#9cded6", "#f0b98d", "#f3e1eb",
]


## T/NK cell data: load and visualise (Fig. 1B, C)

### Load T/NK cell AnnData (with paired metadata and TCR annotation)

In [ ]:
tdata = sc.read('data/06_250529_TDATA_anno2_meta.h5ad')

Filter donors falling below the 5th percentile of recovered cell count to reduce stochastic noise.

In [ ]:
sns.histplot(tdata.obs['PatientID'].value_counts(),bins=500)
#plt.xlim(0,500)
plt.xlabel('T cell number')
plt.ylabel('Patients number')
plt.axvline(x=np.percentile(tdata.obs['PatientID'].value_counts(), 5), color='r')
np.percentile(tdata.obs['PatientID'].value_counts(), 5)

In [ ]:
pco = pd.DataFrame(tdata.obs['PatientID'].value_counts())
ptl = pco[tdata.obs['PatientID'].value_counts()>=633].index.tolist()
print('original {}'.format(len(tdata.obs['PatientID'].unique())))
print('after {}'.format(len(ptl)))

In [ ]:
# Donors with cells in the lowest 5th percentile are excluded from downstream stats
ptl = pco[pco['count'] >= np.percentile(pco['count'], 5)].index.tolist()


In [ ]:
adata = tdata[tdata.obs['PatientID'].isin(ptl)]

In [ ]:
adata = adata[adata.obs['Ethnicity']!='European']

In [ ]:
adata = adata[~adata.obs['Ethnicity'].isna()]

### Per-donor metadata frame

In [ ]:
meta = adata.obs.drop_duplicates('PatientID').set_index('PatientID')[['Age','Sex','BMI','Ethnicity']]

### Apply consistent category orders / colors

In [ ]:
adata.obs['Ethnicity'] = adata.obs['Ethnicity'].cat.reorder_categories(Ethnicity_order)

In [ ]:
adata.obs['anno2'] = adata.obs['anno2'].cat.reorder_categories(T_anno2_order)
adata.uns['anno2_colors'] = T_anno2_colors
adata.uns['Ethnicity_colors'] = Ethnicity_colors


### Fig. 1B — UMAP of T/NK cells colored by subset (`anno2`)

In [ ]:
scjp.us(adata,'anno2', frameon=False, legend_loc='on data',
        legend_fontsize=9, size=0.5)
plt.title('')
scjp.save_fig('Fig1_','T_UMAP_anno2',fig_folder='figures')

### Marker-gene dot plot used to validate T/NK subset annotation

In [ ]:
sc.pl.dotplot(adata,
              ['CD3D','CD4','CD8A','TYROBP','SOX4','CCR7', 'TCF7','CXCR5','ICOS',
               'CXCR3', 
               'IL4R','PTGDR2','GATA3', 'RORC','CCR6','CCR10',
               'CX3CR1', 'ADGRG1','FOXP3','IFI44L','GZMK','GZMB', 'ZNF683','TIGIT','KLRC2','KIR3DL2',
             'TRAV1-2',
              'TRDC','MYB','IL10','CTLA4',
               'NCAM1','FCGR3A','CD27','CD28'], dot_max=0.8,
              groupby=['anno2'], cmap='GnBu',standard_scale='var',
              swap_axes=False,  
              save='T_dotplot.pdf')

## Fig. 1F — Ridge regression of T/NK cell subset frequencies on demographic factors

For each subset we fit a Ridge regression (alpha=1) where outcome is the per-donor proportion of that subset within T/NK cells, and predictors are: age (binarised at the cohort median, 40 y), sex (Female), obesity (BMI ≥ 23 kg/m²) and ethnicity dummy variables. p-values are obtained by 1000-iteration bootstrap.

In [ ]:
test = adata[~adata.obs["BMI"].isna()]

In [ ]:

# ----------------------------------
# donor-level proportion table
# ----------------------------------
pf = pd.crosstab(test.obs['PatientID'], test.obs['anno2'], normalize=0)
pf = pf.reset_index().melt(id_vars=['PatientID']).set_index('PatientID')
pf = pf.merge(meta, left_index=True, right_index=True, how='left')

pf['Ethnicity'] = pf['Ethnicity'].cat.reorder_categories(Ethnicity_order)

# ----------------------------------
# containers
# ----------------------------------
coef_dict = {}
pvalue_dict = {}
ci_low_dict = {}
ci_high_dict = {}
se_dict = {}
r2_dict = {}
n_dict = {}

n_iterations = 1000
alpha = 1

# ----------------------------------
# fit ridge per subset
# ----------------------------------
for a in pf['anno2'].unique():
    subset = pf[pf['anno2'] == a].copy()

    subset['Female'] = (subset['Sex'] == 'Female').astype(int)
    subset['Older (>40)'] = (subset['Age'] >= 40).astype(int)
    subset['Obesity'] = (subset['BMI'] >= 23).astype(int)

    Ethnicity_dummies = pd.get_dummies(subset['Ethnicity'], drop_first=False)

    X = pd.concat([
        subset[['Older (>40)', 'Female', 'Obesity']],
        Ethnicity_dummies
    ], axis=1)

    y = subset['value'].astype(float)
    X_scaled = X.copy()   # 현재 네 최종 코드 유지

    ridge = Ridge(alpha=alpha)
    ridge.fit(X_scaled, y)

    coef = pd.Series(ridge.coef_, index=X.columns)
    coef_dict[a] = coef

    # in-sample R²
    r2_dict[a] = ridge.score(X_scaled, y)
    n_dict[a] = len(y)

    # bootstrap
    coef_bootstrap_list = []
    r2_bootstrap_list = []

    for i in range(n_iterations):
        X_resampled, y_resampled = resample(X_scaled, y, replace=True, random_state=i)
        ridge.fit(X_resampled, y_resampled)
        coef_bootstrap_list.append(ridge.coef_)
        r2_bootstrap_list.append(ridge.score(X_resampled, y_resampled))

    coef_bootstrap = pd.DataFrame(coef_bootstrap_list, columns=X.columns)

    # bootstrap SE
    se_dict[a] = coef_bootstrap.std()

    # empirical bootstrap p-value
    coef_p = {}
    for col in X.columns:
        b = coef_bootstrap[col].values
        p_left = (np.sum(b <= 0) + 1) / (n_iterations + 1)
        p_right = (np.sum(b >= 0) + 1) / (n_iterations + 1)
        coef_p[col] = 2 * min(p_left, p_right)

    pvalue_dict[a] = pd.Series(coef_p)

    # bootstrap percentile CI
    ci_low_dict[a] = coef_bootstrap.quantile(0.025)
    ci_high_dict[a] = coef_bootstrap.quantile(0.975)

# ----------------------------------
# wide tables
# ----------------------------------
codf = pd.DataFrame(coef_dict).T
pvdf = pd.DataFrame(pvalue_dict).T
sedf = pd.DataFrame(se_dict).T
ci_low_df = pd.DataFrame(ci_low_dict).T
ci_high_df = pd.DataFrame(ci_high_dict).T

# align columns
pvdf = pvdf[codf.columns]
sedf = sedf[codf.columns]
ci_low_df = ci_low_df[codf.columns]
ci_high_df = ci_high_df[codf.columns]

# ----------------------------------
# BH correction across all coefficient tests in this panel
# ----------------------------------
pvals_flat = pvdf.values.flatten()
mask = ~np.isnan(pvals_flat)

qvals_flat = np.full_like(pvals_flat, np.nan, dtype=float)
qvals_flat[mask] = multipletests(pvals_flat[mask], method='fdr_bh')[1]

qvdf = pd.DataFrame(qvals_flat.reshape(pvdf.shape), index=pvdf.index, columns=pvdf.columns)

annot_matrix = qvdf.applymap(lambda x: '*' if x <= 0.05 else '')


### Plot ridge coefficient heatmap (Fig. 1F left)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 7)) 
ax = sns.heatmap(codf, annot=annot_matrix, fmt='', cmap='RdBu_r',vmin=-0.036, vmax=0.036,
            linewidths=0.5, linecolor='k',
              cbar_kws={"label": "Ridge Coefficient",
                      "shrink": .5, 'anchor':(-0.3,0), 'aspect':15},)
plt.title('Coefficients from Ridge Regression (p < 0.05)')
plt.xlabel("")
plt.ylabel("")
cbar = fig.figure.get_children()[-1]
cbar.spines[["bottom", "top", "left","right"]].set_visible(True)
cbar.spines[["bottom", "top", "left","right"]].set_color("k")
# cbar.set_ylabel('Jaccard Index', size=15)
plt.tight_layout()
plt.savefig('figures/Fig1_Tcell_ridge_noCLR_FDR.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')

plt.show()

### Build supplementary table of ridge regression results

In [ ]:

# ----------------------------------
# long-format supplementary table
# ----------------------------------
coef_long = codf.stack().rename('coef')
se_long = sedf.stack().rename('bootstrap_se')
ci_low_long = ci_low_df.stack().rename('ci_low')
ci_high_long = ci_high_df.stack().rename('ci_high')
p_long = pvdf.stack().rename('p_value_raw')
q_long = qvdf.stack().rename('p_value_fdr')

supp_df = pd.concat(
    [coef_long, se_long, ci_low_long, ci_high_long, p_long, q_long],
    axis=1
).reset_index()

supp_df.columns = ['subset', 'predictor', 'coef', 'bootstrap_se', 'ci_low', 'ci_high', 'p_value_raw', 'p_value_fdr']

# add model-level metrics
r2_series = pd.Series(r2_dict, name='r2')
n_series = pd.Series(n_dict, name='n_donors')

supp_df = supp_df.merge(r2_series, left_on='subset', right_index=True, how='left')
supp_df = supp_df.merge(n_series, left_on='subset', right_index=True, how='left')

supp_df['significant_fdr'] = supp_df['p_value_fdr'] < 0.05

# optional: nicer ordering
predictor_order = ['Older (>40)', 'Female', 'Obesity'] + list(Ethnicity_order)
supp_df['predictor'] = pd.Categorical(supp_df['predictor'], categories=predictor_order, ordered=True)
supp_df = supp_df.sort_values(['subset', 'predictor']).reset_index(drop=True)

In [ ]:
supp_df.to_excel('Sup_ridge_T_table.xlsx', index=False)

### Fig. 1F right — TCR/BCR reconstruction status stacked by subset is built downstream alongside the BCR equivalent

## Fig. 1G — T/NK subset proportions stratified by age group and sex

In [ ]:
iso[['Sex','Age_group']] = iso['Age_Sex_group'].str.split('_', expand=True)
iso['Age_group'] = iso['Age_group'].astype(int)

In [ ]:
iso = iso[iso.anno2.str.startswith("T")]

In [ ]:

# 2) 전체 남녀 합친 Age_group별 mean과 SEM 계산
stats_df = (
    iso
    .groupby(['Age_group','anno2'])['value']
    .agg(mean='mean', sem='sem')
    .reset_index()
)

# 3) barplot + facetgrid
g = sns.catplot(
    data=iso,
    x='Age_group',
    y='value',
    hue='Sex',
    col='anno2',
    col_wrap=6,
    kind='bar',
    dodge=True,
    errorbar='se',
    capsize=0.05,
    errwidth=1,
    palette=['orangered','mediumseagreen'],  # Female, Male
    height=3,
    aspect=0.8,
        sharey=False, edgecolor='k',
    alpha=0.7, legend=False
)

# 4) facet마다 평균선(에러바 포함) 추가
handles, labels = [], []
first = True
for ax in g.axes.flatten():
    cell = ax.get_title().split(' = ')[-1]
    df_tmp = stats_df[stats_df['anno2']==cell]
    xs = [0, 1]  # <40, ≥40
    ys = df_tmp['mean'].values
    errs = df_tmp['sem'].values

    ax.errorbar(
        xs, ys,
        yerr=errs,
        marker='o',
        linestyle='-',
        color='black',
        capsize=3,
        elinewidth=1,
        label='Overall mean ± SE' if first else ""
    )
    if first:
        ax.legend(loc='upper right', fontsize='small', frameon=False)
        first = False

# 5) 축·타이틀·레이아웃 정리
g.set_axis_labels("Age group", "Proportion")
g.set_titles("{col_name}")
for ax in g.axes.flatten():
    ax.set_xticks([0,1])
    ax.set_xticklabels(['<40','≥40'])
    sns.despine(ax=ax)
    
 

plt.tight_layout()
plt.show()

# B cell analyses (Fig. 1D, E, H, I)

### Load B cell AnnData (with paired metadata and BCR annotation)

In [ ]:
bdata = sc.read('data/06_250608_BDATA_IGLK_annotated_meta.h5ad')

In [ ]:
sns.histplot(bdata.obs['PatientID'].value_counts(),bins=500)
plt.xlim(0,500)
plt.xlabel('B cell number')
plt.ylabel('Patients number')
plt.axvline(x=39, color='r')

In [ ]:
np.percentile(bdata.obs['PatientID'].value_counts(), 5)

In [ ]:
np.percentile(bdata.obs['PatientID'].value_counts(), 10)

In [ ]:
pco = pd.DataFrame(bdata.obs['PatientID'].value_counts())
ptl = pco[bdata.obs['PatientID'].value_counts()>39].index.tolist()
print('original {}'.format(len(bdata.obs['PatientID'].unique())))
print('after {}'.format(len(ptl)))

In [ ]:
# Donors with very few B cells (5th percentile or lower) excluded
ptl = pco[pco['count'] >= np.percentile(pco['count'], 5)].index.tolist()


In [ ]:
adata = bdata[bdata.obs['PatientID'].isin(ptl)]

In [ ]:
adata = adata[~adata.obs['Ethnicity'].isna()]

In [ ]:
adata.obs['anno2'] = np.where(adata.obs['anno2']=='B_Plasmablast', 'B_Plasma', adata.obs['anno2'])

In [ ]:
meta = adata.obs.drop_duplicates('PatientID').set_index('PatientID')[['Age','Sex','Ethnicity','BMI']]

### Apply consistent category orders / colors for B cells

In [ ]:
Ethnicity_order=['Chinese', 'Malay','Indian','Japanese','Korean','Thai']


anno1_order = ['B_Naive', 'B_Memory', 'B_Plasma']
anno2_order = ['B_Naive_transitional', 'B_Naive', 'B_Naive_IFN', 'B_Memory_IFN',
       'B_Memory_unswitched','B_Memory_atypical', 'B_Memory_switched', 
        'B_Plasma']


sex_colors =  ["#87CEFA", "#FB7C7C"]

age_colors = ["#E5F5E0", "#74C476", "#238B45", "#00441B"]

shm_colors = ['grey','steelblue', 'mediumblue']

Ethnicity_colors = ["tomato",'gold',  "dodgerblue",  "limegreen",  "aquamarine", "orchid",  'gray']

adata.uns['Ethnicity_colors'] = ["tomato",'gold',  "dodgerblue",  "limegreen",  "aquamarine", "orchid",  'gray']

adata.uns['anno1_colors'] = ['dodgerblue',
 'seagreen', 'indianred'] 

adata.uns['anno2_colors'] = ['#a6cee3', '#1f78b4', 'goldenrod', 'darkgoldenrod', 'limegreen','saddlebrown',
       '#33a02c',  '#e31a1c']


In [ ]:
adata.obs['Ethnicity'] = adata.obs['Ethnicity'].cat.reorder_categories(Ethnicity_order)

In [ ]:
adata.obs['anno2'] = adata.obs['anno2'].astype('category')
adata.obs['anno2'] = adata.obs['anno2'].cat.reorder_categories(['B_Naive_transitional', 'B_Naive', 'B_Naive_IFN', 'B_Memory_IFN',
       'B_Memory_unswitched', 'B_Memory_atypical', 'B_Memory_switched',
        'B_Plasma'])

### Fig. 1D — UMAP of B cells colored by subset (`anno2`)

In [ ]:
scjp.us(adata,'anno2', frameon=False, legend_loc='on data',
        legend_fontsize=9, size=3)
plt.title('')
scjp.save_fig('Fig1_','B_UMAP_anno2',fig_folder='figures')

### Marker-gene dot plot used to validate B cell annotation

In [ ]:
sc.pl.dotplot(adata, var_names='CD19,MS4A1,SOX4,CD27,NT5E,IFI44L,IGHD,IGHM,FCRL5,TBX21,CD38,PRDM1,XBP1,SDC1'.split(','),
             groupby='anno2', 
              dot_max=0.8,standard_scale='var',cmap='GnBu', 
              save='B_dotplot.pdf')
#TYMS,CDK1,

## Fig. 1H — Ridge regression of B cell subset frequencies on demographic factors

In [ ]:
test = adata[~adata.obs["BMI"].isna()]

In [ ]:
test = test[test.obs['Country']!='IN']

In [ ]:



# ----------------------------------
# donor-level proportion table
# ----------------------------------
pf = pd.crosstab(test.obs['PatientID'], test.obs['anno2'], normalize=0)
pf = pf.reset_index().melt(id_vars=['PatientID']).set_index('PatientID')
pf = pf.merge(meta, left_index=True, right_index=True, how='left')

pf['Ethnicity'] = pf['Ethnicity'].cat.reorder_categories(Ethnicity_order)

# ----------------------------------
# containers
# ----------------------------------
coef_dict = {}
pvalue_dict = {}
ci_low_dict = {}
ci_high_dict = {}
se_dict = {}
r2_dict = {}
n_dict = {}

n_iterations = 1000
alpha = 1

# ----------------------------------
# fit ridge per subset
# ----------------------------------
for a in pf['anno2'].unique():
    subset = pf[pf['anno2'] == a].copy()

    subset['Female'] = (subset['Sex'] == 'Female').astype(int)
    subset['Older (>40)'] = (subset['Age'] >= 40).astype(int)
    subset['Obesity'] = (subset['BMI'] >= 23).astype(int)

    Ethnicity_dummies = pd.get_dummies(subset['Ethnicity'], drop_first=False)

    X = pd.concat([
        subset[['Older (>40)', 'Female', 'Obesity']],
        Ethnicity_dummies
    ], axis=1)

    y = subset['value'].astype(float)
    X_scaled = X.copy()   # 현재 네 최종 코드 유지

    ridge = Ridge(alpha=alpha)
    ridge.fit(X_scaled, y)

    coef = pd.Series(ridge.coef_, index=X.columns)
    coef_dict[a] = coef

    # in-sample R²
    r2_dict[a] = ridge.score(X_scaled, y)
    n_dict[a] = len(y)

    # bootstrap
    coef_bootstrap_list = []
    r2_bootstrap_list = []

    for i in range(n_iterations):
        X_resampled, y_resampled = resample(X_scaled, y, replace=True, random_state=i)
        ridge.fit(X_resampled, y_resampled)
        coef_bootstrap_list.append(ridge.coef_)
        r2_bootstrap_list.append(ridge.score(X_resampled, y_resampled))

    coef_bootstrap = pd.DataFrame(coef_bootstrap_list, columns=X.columns)

    # bootstrap SE
    se_dict[a] = coef_bootstrap.std()

    # empirical bootstrap p-value
    coef_p = {}
    for col in X.columns:
        b = coef_bootstrap[col].values
        p_left = (np.sum(b <= 0) + 1) / (n_iterations + 1)
        p_right = (np.sum(b >= 0) + 1) / (n_iterations + 1)
        coef_p[col] = 2 * min(p_left, p_right)

    pvalue_dict[a] = pd.Series(coef_p)

    # bootstrap percentile CI
    ci_low_dict[a] = coef_bootstrap.quantile(0.025)
    ci_high_dict[a] = coef_bootstrap.quantile(0.975)

# ----------------------------------
# wide tables
# ----------------------------------
codf = pd.DataFrame(coef_dict).T
pvdf = pd.DataFrame(pvalue_dict).T
sedf = pd.DataFrame(se_dict).T
ci_low_df = pd.DataFrame(ci_low_dict).T
ci_high_df = pd.DataFrame(ci_high_dict).T

# align columns
pvdf = pvdf[codf.columns]
sedf = sedf[codf.columns]
ci_low_df = ci_low_df[codf.columns]
ci_high_df = ci_high_df[codf.columns]

# ----------------------------------
# BH correction across all coefficient tests in this panel
# ----------------------------------
pvals_flat = pvdf.values.flatten()
mask = ~np.isnan(pvals_flat)

qvals_flat = np.full_like(pvals_flat, np.nan, dtype=float)
qvals_flat[mask] = multipletests(pvals_flat[mask], method='fdr_bh')[1]

qvdf = pd.DataFrame(qvals_flat.reshape(pvdf.shape), index=pvdf.index, columns=pvdf.columns)

annot_matrix = qvdf.applymap(lambda x: '*' if x <= 0.05 else '')




In [ ]:
fig, ax = plt.subplots(figsize=(7,5)) 
ax = sns.heatmap(codf, annot=annot_matrix, fmt='', cmap='RdBu_r',vmin=-0.046, vmax=0.046,
            linewidths=0.5, linecolor='k',
              cbar_kws={"label": "Ridge Coefficient",
                      "shrink": .5, 'anchor':(-0.3,0), 'aspect':15},)
plt.title('Coefficients from Ridge Regression (p < 0.05)')
plt.xlabel("")
plt.ylabel("")
cbar = fig.figure.get_children()[-1]
cbar.spines[["bottom", "top", "left","right"]].set_visible(True)
cbar.spines[["bottom", "top", "left","right"]].set_color("k")
# cbar.set_ylabel('Jaccard Index', size=15)
plt.tight_layout()
plt.savefig('figures/Fig1_Bcell_ridge_noCLR_FDR.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')

plt.show()


In [ ]:


# ----------------------------------
# long-format supplementary table
# ----------------------------------
coef_long = codf.stack().rename('coef')
se_long = sedf.stack().rename('bootstrap_se')
ci_low_long = ci_low_df.stack().rename('ci_low')
ci_high_long = ci_high_df.stack().rename('ci_high')
p_long = pvdf.stack().rename('p_value_raw')
q_long = qvdf.stack().rename('p_value_fdr')

supp_df = pd.concat(
    [coef_long, se_long, ci_low_long, ci_high_long, p_long, q_long],
    axis=1
).reset_index()

supp_df.columns = ['subset', 'predictor', 'coef', 'bootstrap_se', 'ci_low', 'ci_high', 'p_value_raw', 'p_value_fdr']

# add model-level metrics
r2_series = pd.Series(r2_dict, name='r2')
n_series = pd.Series(n_dict, name='n_donors')

supp_df = supp_df.merge(r2_series, left_on='subset', right_index=True, how='left')
supp_df = supp_df.merge(n_series, left_on='subset', right_index=True, how='left')

supp_df['significant_fdr'] = supp_df['p_value_fdr'] < 0.05

# optional: nicer ordering
predictor_order = ['Older (>40)', 'Female', 'Obesity'] + list(Ethnicity_order)
supp_df['predictor'] = pd.Categorical(supp_df['predictor'], categories=predictor_order, ordered=True)
supp_df = supp_df.sort_values(['subset', 'predictor']).reset_index(drop=True)

supp_df.to_excel('Sup_ridge_B_table.xlsx', index=False)

### Catplot of B cell proportions by ethnicity (sanity check)

In [ ]:
fordata = adata[(adata.obs['Age']>30) & (adata.obs['Age']<60)]
iso = pd.crosstab([fordata.obs['PatientID'], fordata.obs['Ethnicity']], fordata.obs['anno2'], normalize=0) 
iso = iso.reset_index().melt(id_vars=['PatientID','Ethnicity']) 
iso = iso.merge(meta[['Age','Sex']], left_on='PatientID', right_on='PatientID', how='left')
plt.figure(figsize=(8,6))

g = sns.barplot(data=iso, x='anno2', y='value', hue='Ethnicity', 
                hue_order=Ethnicity_order,
             palette=Ethnicity_colors, alpha=0.6,
            errcolor='black', ec='black', errwidth=1,  errorbar='se',
capsize=0.05,)

# sns.stripplot(data=iso, y='anno2', x='value', hue='Sex', 
#                 hue_order=['Male','Female'],
#              palette=sex_colors, alpha=0.5, ec='k', linewidth=0.5, s=2, dodge=True)
# g.set_ylim([0,0.6])
# plt.title('Isotype proportion in memory B cell')
plt.xlabel('')
plt.ylabel('Proportion')
sns.despine()
plt.xticks(rotation=90)
# g.get_legend().remove()
plt.legend(loc='upper right', edgecolor='black', fontsize='small')

# for i,f in enumerate(iso['anno2'].unique().tolist()):
#     df1=iso[iso['anno2']==f]
#     p1 = stats.ttest_ind(df1[df1['Sex']=='Male']['value'],df1[df1['Sex']=='Female']['value'])[1]
#     ap = ols("value ~ Sex + Age + Ethnicity", data = df1).fit()
#     if p1 < 0.05 :
#         up = stats.t.interval(alpha=0.95, df=len(df1['value'])-1, loc=np.mean(df1['value']), scale=stats.sem(df1['value']))[1]
#         plt.text(y=i-0.3,x=up*1.2,s='p={:.3f}'.format(p1),fontsize=10)
#         plt.text(y=i-0.3, x=up*1.5,s='p={:.3f}'.format(ap.pvalues[1]),fontsize=10, color='blue')

In [ ]:
g = sns.catplot(
    data=iso,
    x="Ethnicity",
    y="value",
    col="anno2",
    col_order=anno2_order,
    col_wrap=4,           # wrap into 3 columns → 7 rows for 20 cell types
    sharey=False,         # each facet has its own y‐axis scale
    kind="bar",
    palette=Ethnicity_colors,
    order=Ethnicity_order,
    errorbar="se",
    capsize=0.1,
    height=4,           # height of each facet (inches)
    aspect=0.9, 
    alpha=0.6,
     errwidth=1.5,
    ec='k' 
)

# 4) Tweak the axis labels, rotate x‐tick labels, and add a super‐title:
g.set_axis_labels("", "Proportion")
g.set_titles("{col_name}")    # facet title = the anno2 name

for ax in g.axes.flatten():
    # Rotate the x‐axis tick labels so the ethnicity names don’t overlap
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=6)

# 5) Move / style the legend:
#    By default, catplot will show one legend for Ethnicity. 
#    We can move it to the bottom center, for example:
# plt.legend(
#     title="Ethnicity",
#     bbox_to_anchor=(0.5, -0.02),
#     loc="upper center",
#     ncol=3,
#     frameon=False,
# )

# 6) Final layout adjustments:
sns.despine(left=True)
g.fig.subplots_adjust(
    top=0.9,    # leave room for the suptitle
    hspace=0.3,  # increase vertical spacing between rows
    wspace=0.5   # horizontal spacing between columns
)
g.fig.suptitle("Cell‐type Proportion by Ethnicity (Age 30–60)", fontsize=14)
plt.savefig('figures/Fig1_Bcell_facet.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')
plt.show()

## Fig. 1I — B cell subset proportions stratified by age group and sex

In [ ]:
fordata = adata[~adata.obs['anno2'].str.endswith('IFN')]
fordata.obs['Age_group'] = (fordata.obs['Age'] >= 40).astype(int)
fordata.obs['Sex'] = fordata.obs['Sex'].astype(str) 
fordata.obs['Age_Sex_group'] =  fordata.obs['Sex'] + '_' + fordata.obs['Age_group'].astype(str)

## Age-stratified composition_Sex_group 기준으로 데이터 정리
iso = pd.crosstab([fordata.obs['PatientID'], fordata.obs['Age_Sex_group']], fordata.obs['anno2'], normalize=0)
iso = iso.reset_index().melt(id_vars=['PatientID', 'Age_Sex_group'])
iso[['Sex','Age_group']] = iso['Age_Sex_group'].str.split('_', expand=True)
iso['Age_group'] = iso['Age_group'].astype(int)
 

In [ ]:
def custom_formatter(x, pos):
    s = str(x)
    if '.' in s:
        frac = s.split('.', 1)[1]
        if len(frac) > 4:
            return f"{x:.3f}"
    return s
# 2) 전체 남녀 합친 Age_group별 mean과 SEM 계산
stats_df = (
    iso
    .groupby(['Age_group','anno2'])['value']
    .agg(mean='mean', sem='sem')
    .reset_index()
)

# 3) barplot + facetgrid
g = sns.catplot(
    data=iso,
    x='Age_group',
    y='value',
    hue='Sex',
    col='anno2',
    col_wrap=6,
    kind='bar',
    dodge=True,
    errorbar='se',
    capsize=0.05,
    errwidth=1,
    palette=['lightcoral','lightseagreen'],  # Female, Male
    height=4,
    aspect=0.8,
        sharey=False, edgecolor='k',
    alpha=0.7, legend=False
)

# 4) facet마다 평균선(에러바 포함) 추가
handles, labels = [], []
first = True
for ax in g.axes.flatten():
    cell = ax.get_title().split(' = ')[-1]
    df_tmp = stats_df[stats_df['anno2']==cell]
    xs = [0, 1]  # <40, ≥40
    ys = df_tmp['mean'].values
    errs = df_tmp['sem'].values

    ax.errorbar(
        xs, ys,
        yerr=errs,
        marker='o',
        linestyle='--', linewidth=2,
        color='black',
        capsize=3,
        elinewidth=1,
        label='Overall mean ± SE' if first else ""
    )
    if first:
        ax.legend(loc='upper left', fontsize='small', frameon=False)
        first = False

# 5) 축·타이틀·레이아웃 정리
g.set_axis_labels("Age group", "Proportion")
g.set_titles("{col_name}")

# ① 한 번 그림을 그려서 tick label을 업데이트
g.fig.canvas.draw()

for ax in g.axes.flatten():
    ax.set_xticks([0,1])
    ax.set_xticklabels(['<40','≥40'])
    sns.despine(ax=ax)

    # ② 현재 y축 major tick label 문자열 얻기
    labels = [t.get_text() for t in ax.get_yticklabels()]
    # ③ 소수점 이하 4자리 초과가 하나라도 있으면 3자리로 포맷
    if any(
        ('.' in lab and len(lab.split('.',1)[1]) > 3)
        for lab in labels
    ):
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

plt.tight_layout()
plt.savefig('figures/Fig1_B_composition_age_sex_group_barplot.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')

plt.show()

In [ ]:
def custom_formatter(x, pos):
    s = str(x)
    if '.' in s:
        frac = s.split('.', 1)[1]
        if len(frac) > 4:
            return f"{x:.3f}"
    return s

# 2) 전체 남녀 합친 Age_group별 mean과 SEM 계산 (그대로 유지: y축 포맷팅 판단에 쓸 수 있음)
stats_df = (
    iso
    .groupby(['Age_group','anno2'])['value']
    .agg(mean='mean', sem='sem')
    .reset_index()
)

# 3) barplot + facetgrid (legend는 나중에 pointplot에서 추가)
g = sns.catplot(
    data=iso,
    x='Age_group',
    y='value',
    hue='Sex',
    col='anno2',
    col_wrap=6,
    kind='bar',
    dodge=True,
    errorbar='se',
    capsize=0.05,
    errwidth=1,
    palette=['lightcoral','lightseagreen'],  # Female, Male
    height=3,
    aspect=0.8,
    sharey=False, edgecolor='k',
    alpha=0.7, legend=False
)

# 4) 성별별 Young→Old 추세선(바 위에)만 오버레이
#    errorbar=None 로 깔끔하게 선만, dodge로 막대 중심에 sort
g.map_dataframe(
    sns.pointplot,
    x='Age_group', y='value', hue='Sex',
    dodge=0.38,       # 막대폭에 맞춰 적당히
    markers='.',
    linestyles='--',
    errorbar=None,    # 추세선만
    palette=['indianred','seagreen']
)

# FacetGrid에 하나의 범례만 추가
g.add_legend(title='Sex')

# 5) 축·타이틀·레이아웃 정리
g.set_axis_labels("Age group", "Proportion")
g.set_titles("{col_name}")

# 한 번 그림을 그려서 tick label을 업데이트
g.fig.canvas.draw()

for ax in g.axes.flatten():
    ax.set_xticks([0,1])
    ax.set_xticklabels(['<40','≥40'])
    sns.despine(ax=ax)

    # 현재 y축 major tick label 문자열 얻기
    labels = [t.get_text() for t in ax.get_yticklabels()]
    # 소수점 이하 4자리 초과가 하나라도 있으면 3자리로 포맷
    if any(
        ('.' in lab and len(lab.split('.',1)[1]) > 3)
        for lab in labels
    ):
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

plt.tight_layout()
plt.savefig('figures/Fig1_B_composition_age_sex_group_barplot.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')
plt.show()

### B cell subset proportion by age group heatmap

In [ ]:
fordata = adata.copy()
iso = pd.crosstab(fordata.obs['PatientID'], fordata.obs['anno2'], normalize=0) 

epsilon = 1e-10
prop = iso + epsilon
prop = prop.div(prop.sum(axis=1), axis=0)
gm = np.exp(np.log(prop).mean(axis=1))
#    CLR: log(y_ij / gm_i)
clr = np.log(prop.div(gm, axis=0))
iso = clr.reset_index().melt(id_vars='PatientID')

iso = iso.merge(meta, left_on='PatientID', right_on='PatientID', how='left')

# iso = iso[iso.value>-14]

for a in iso.anno2.unique():
    print(a)
    
    aso = iso[iso.anno2==a]
    print(stats.pearsonr(aso['Age'], aso['value']))
    g = sns.lmplot(data=aso, x="Age", y="value", 
          hue='Sex', hue_order=['Male','Female'], palette=sex_colors,
              scatter_kws={ "ec":"black", "s":30, "alpha":0.6}, line_kws={"lw":3, "ls":"--","alpha":0.8})
    for ax in g.axes.flat:
        for line in ax.lines:
            line.set_path_effects([path_effects.Stroke(linewidth=5, foreground='black'),
                               path_effects.Normal()])

    plt.ylabel('Proportion')
    plt.title(a)
    plt.show()
   


In [ ]:
from sklearn.preprocessing import MaxAbsScaler as mabs

iso = pd.crosstab(adata.obs['PatientID'], adata.obs['anno2'], normalize=0)  
iso = iso.merge(meta[['Age']], left_on='PatientID', right_on='PatientID', how='left')

iso['Age_Group'] = pd.cut(iso['Age'], bins=[19, 30, 40, 50, 60, 70], labels=['20-30', '31-40', '41-50', '51-60', '61-70'])

df = iso.groupby('Age_Group').mean()

df = pd.DataFrame(mabs().fit_transform(df), index = df.index, columns=df.columns)

from matplotlib.colors import LinearSegmentedColormap
colors = ['#FFFFFF', '#FFF7DC', '#FFDAB9', '#FFB6C1', 'crimson']
n_bins = 500
cmap = LinearSegmentedColormap.from_list('custom_pastel', colors, N=n_bins,
                                        )

n_individuals = len(adata.obs['PatientID'].unique())
ages = adata.obs['Age']               
t_cell_subsets = adata.obs['anno2'].cat.categories
 
# Build DataFrame
df = df.copy()
del df['Age']
# Build age group bins
#df['Age_Group'] = pd.cut(df['Age'], bins=[19, 30, 40, 50, 60, 70], labels=['20-30', '31-40', '41-50', '51-60', '61-70'])

# Mean per age group
#age_group_means = df.groupby('Age_Group')[t_cell_subsets].mean()



# Plot heatmap
plt.figure(figsize=(5, 5))
ax = sns.heatmap(df.T,cmap=cmap, linewidths=0.5, linecolor='k', 
                         cbar_kws={"label": "Scaled proportion",
                      "shrink": .5, },
           vmin=0, alpha=0.7)

plt.title('B Cell Subset Proportions by Age Group', fontsize=16)
#plt.xlabel('T Cell Subsets', fontsize=12)
plt.xlabel('Age Groups', fontsize=12)

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
cbar = ax.collections[0].colorbar

# 3) colorbar 테두리(외곽선) 설정
cbar.outline.set_edgecolor('black')  # Border color
cbar.outline.set_linewidth(1)  
plt.tight_layout()
plt.savefig('figures/Fig1_B_age_heatmap.pdf'.format(a), dpi=300, format='pdf',transparent=True, bbox_inches='tight')
plt.show()